# SXF — Isolated GPU Benchmark V2

Measures **Qwen3 8B** and **Qwen3-VL 4B**, one at a time, on a real Google Colab NVIDIA GPU. Does not run inference on your MacBook.

Choose **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. If no free GPU is assigned, the notebook stops; do not pay for an upgrade unless you choose to.

This procedure clears loaded Ollama models before each capture, records the real response to a test image, GPU device-wide usage, Ollama runtime-reported residency, exact model digest and per-trial output. It does not claim independent model-only VRAM, complete GPU residency, or guaranteed quality. The ZIP stays unpublished pending review.

**Important:** This reruns both models once for a cleaner comparison. Earlier evidence remains preserved separately.

In [ ]:
import pathlib,subprocess,sys,shutil,importlib
ROOT=pathlib.Path("/content/sxf-ai")
OUTPUT=pathlib.Path("/content/sxf-isolated-gpu-v2")
OUTPUT.mkdir(parents=True,exist_ok=True)
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No NVIDIA GPU assigned in Colab. Select T4 GPU in Runtime settings or retry when it is available; no benchmark was run.")
if not (ROOT/".git").is_dir():
    subprocess.run(["git","clone","--depth","1","--branch","main","https://github.com/rab2323s-creator/sxf-ai.git",str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"fetch","origin","main"],check=True)
    subprocess.run(["git","-C",str(ROOT),"merge","--ff-only","FETCH_HEAD"],check=True)
sys.path.insert(0,str(ROOT/"scripts"))
import ensure_colab_ollama
importlib.reload(ensure_colab_ollama)
OLLAMA=ensure_colab_ollama.ensure_ollama_ready(OUTPUT)
print("GPU assigned and Ollama ready:",OLLAMA)

In [ ]:
import subprocess,sys,json
run=subprocess.run([sys.executable,"-u","scripts/measure_isolated_gpu_evidence.py",
    "--output-dir",str(OUTPUT),"--ollama-bin",OLLAMA,"--context","4096","--repeats","3"],
    cwd=ROOT,capture_output=True,text=True)
if run.stdout:print(run.stdout[-20000:],flush=True)
if run.stderr:print(run.stderr[-20000:],flush=True)
if run.returncode:raise RuntimeError("Evidence capture failed. Original error shown above. Exit="+str(run.returncode))
for path in sorted(OUTPUT.glob("*-isolated.json")):
    record=json.loads(path.read_text())
    print(record["modelTag"],"trial tok/s:",[x["tokensPerSecond"] for x in record["runRecords"] if not x["warmup"]])
    if record["workload"]=="vision":
        print("Vision answer (manual review required):",record["runRecords"][-1]["answer"])

In [ ]:
import pathlib,zipfile,json,hashlib,datetime
from google.colab import files
names=["qwen3-8b-isolated.json","qwen3-vl-4b-isolated.json","vision-square.png"]
for name in names:
    if not (OUTPUT/name).is_file():raise RuntimeError("Missing required measured file: "+name)
manifest={"kind":"sxf-isolated-gpu-evidence","reviewStatus":"manual-review-required",
          "createdAtUtc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
          "files":[{"name":name,"sha256":hashlib.sha256((OUTPUT/name).read_bytes()).hexdigest()} for name in names]}
(OUTPUT/"manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
archive=pathlib.Path("/content/sxf-isolated-gpu-evidence-v2.zip")
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for name in names+["manifest.json"]:z.write(OUTPUT/name,name)
print("Evidence ready, unpublished:",archive)
files.download(str(archive))